# 01 – Load and explore
**Goal:** understand the dataset and flag data-quality problems *before* modelling.
Write what you see and why you decide things in `docs/decisions_log.md` as you go.

In [ ]:
import sys; sys.path.insert(0, "../src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from ev_energy import load_data, quality_report, TARGET

df = load_data("../data/raw/ev_trip_energy.csv")
print(df.shape)
df.head()

## 1. Structure, types, missing values, duplicates

In [ ]:
print(df.dtypes)
print("missing:", int(df.isna().sum().sum()), "| duplicate rows:", int(df.duplicated().sum()))
print("vehicles:", df.vehicle_id.nunique(), "| trips per vehicle (min/median/max):",
      df.vehicle_id.value_counts().agg(["min","median","max"]).tolist())
df.describe().T

> **Identifying our values**
>
> **Structure.** The table has 10,151 trips from 50 vehicles and no missing values or duplicate rows, so nothing needs filling in or de-duplicating.
> Vehicles are unbalanced: one has only 38 trips and another has 372 (median about 204). This is why the train/test split is done by vehicle, so one heavily represented car cannot dominate either set.
> The summary table shows impossible values: raw Speed up to 930, latitude down to -645, and total voltage up to 435 V when the typical value is about 305 V.

## 2. Plausibility checks (flag, don't delete)

In [ ]:
quality_report(df)

> **Flagging values that are unrealistic**
>
> **Plausibility check.** After applying the ranges in `PLAUSIBLE_RANGES`, the problem rows are concentrated in a few columns:
> - `speed_raw`: 1,009 rows above 150 km/h (and 285 rows exactly 0, even though every trip is at least 6 km).
> - `lat` and `lon`: 11 rows each with impossible coordinates, from 10 vehicles.
> - `voltage_v`: 1 row at 435 V.
> - Energy, distance, hour, weekday, temperatures and duration: no rows outside their ranges.
>
> These checks flag rows rather than deleting them, because an extreme value might be a real long trip. The cleaning rules decided from this are: drop `speed_raw`, `lat` and `lon` as columns, remove the single 435 V row, and keep everything else.

**Things to investigate and explain in the report**
- `speed_raw` reaches ~930 –  it is said to be km/hr so this is impossible. WE can compare with `avg_speed_kmh = distance / duration` (below).
- Voltage, latitude, longitude have impossible extremes – sensor glitches or real?
- Decide per column: remove, cap, or keep (and justify).

In [ ]:
ratio = df.speed_raw / df.avg_speed_kmh
print(ratio.describe())
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(df.avg_speed_kmh, df.speed_raw, s=3, alpha=.3)
ax[0].set(xlabel="distance/duration (km/h)", ylabel="raw Speed column", title="Is Speed really km/h?")
ax[1].hist(df.avg_speed_kmh, bins=60); ax[1].set(xlabel="avg speed (km/h)", title="Derived average speed")
plt.tight_layout(); plt.savefig("../figures/speed_check.png", dpi=150); plt.show()

> **DRAFT interpretation (written by Claude; edit in your own words and check it against your own output):**
>
> **Is the raw Speed column km/h?** The dataset documentation says it is the average speed in km/h, but it does not behave that way. Dividing each raw value by distance/duration gives ratios from 0 up to about 237 (median about 2), so the two are not proportional, and 285 trips have a raw speed of exactly 0. The derived speed (distance divided by duration) never exceeds 90 km/h, which is plausible for a small city car.
> **Decision:** do not use `speed_raw`; use `avg_speed_kmh = distance / duration` instead. (Describe what your own scatter plot shows here: points that do not follow a straight line support this decision.)

## 3. Target variable

In [ ]:
e = df[TARGET]
print("unique values:", e.nunique())
print("smallest steps between values:", np.unique(np.diff(np.sort(e.unique())).round(4))[:5])
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].hist(e, bins=60); ax[0].set(xlabel="Trip energy (kWh)", title="Target distribution")
ax[1].hist(np.log1p(e), bins=60); ax[1].set(xlabel="log(1+energy)", title="Log-transformed")
plt.tight_layout(); plt.savefig("../figures/target_dist.png", dpi=150); plt.show()

> **Interpretation**
>
> **Target variable.** Trip energy is right-skewed (mean 3.95 kWh, median 3.14 kWh, maximum 20.8 kWh): most trips are short and use little energy, with a long tail of high-energy trips. The log-transformed histogram is closer to symmetric.
> Only 83 distinct values occur, all multiples of 0.224 kWh (for example 0.672 and 0.896). This suggests the energy was recorded in coarse steps, so there is a floor on how exact any prediction can be, and the predicted-vs-actual plots will show horizontal banding. This is a limitation to state in the report.

**Observation to discuss:** the target takes only a few dozen discrete values (multiples of 0.224 kWh).
This limits the best achievable error and R² – mention it under limitations, and check how the dataset describes it
(see the *Data Replication Explanatory File*).

## 4. Relationships with energy

In [ ]:
num = ["distance_km","duration_min","avg_speed_kmh","speed_raw","current_a","voltage_v","temp_max_c","temp_min_c","hour","weekday"]
corr = df[[TARGET]+num].corr(method="spearman")[TARGET].drop(TARGET).sort_values()
print(corr.round(3))
fig, axs = plt.subplots(2, 3, figsize=(13, 7))
for ax, c in zip(axs.ravel(), ["distance_km","duration_min","avg_speed_kmh","current_a","voltage_v","temp_max_c"]):
    ax.scatter(df[c], df[TARGET], s=2, alpha=.25); ax.set(xlabel=c, ylabel="energy (kWh)")
plt.tight_layout(); plt.savefig("../figures/energy_vs_features.png", dpi=150); plt.show()


> **Relationships with energy (Spearman rank correlation).**
> - Distance is by far the strongest: 0.92.
> - Average speed (0.42), duration (0.30) and current (0.19) are weaker but positive.
> - Voltage is slightly negative (-0.11); hour, weekday and the temperatures are close to zero (below about 0.07).
>
> distance alone explains most of the variation, how much can the other variables add beyond a fixed kWh-per-km rate?

## 5. Leakage and multicollinearity 
- `current_a` and `voltage_v` are measured *during* the trip, so they are not known in advance. 
- Max/min cell temperature and distance/duration/speed are strongly related. 

In [ ]:
df[num].corr().round(2)


>
> **Collinearity and leakage.**
> - `temp_max_c` and `temp_min_c` are almost the same variable (correlation 0.99), so they carry nearly identical information. This matters for interpreting feature importance, because the credit will be split between them.
> - Duration and average speed are linked by construction (correlation -0.58), and both come from distance, so they overlap with it.
> - Current and voltage are measured during the trip, so they would not be known in advance. Comparing trip-only features against trip + electrical features shows how much a real pre-trip predictor would lose.
> - Current is negative on 280 trips, probably regenerative braking returning energy, an assumption that we make
